# HPM — Phase A: CelebA Supervised-Contrastive Training (Colab)

Milestone 1 (identity-only, contrastive) for the Hemispheric Perception Model.
Trains the two-stream HPM (CNN local + ViT global + callosum + right-dominant
fusion) with **SupCon** on CelebA.

Features:
- **Ultralytics-style** progress bar + `best.pt` / `last.pt` checkpoints.
- **Auto-resume**: rerun the training cell after a Colab disconnect and it
  continues from the last saved epoch (RNG, optimizer, scheduler, AMP all restored).
- Checkpoints + identity splits live on Google Drive so they survive disconnects.

Run the cells top-to-bottom. The only thing you must provide is a Kaggle API token
(`kaggle.json`) and a copy of this repo (on Drive or GitHub).

## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 2. Mount Google Drive

Checkpoints and the identity-split files are written here so they persist across
sessions. Edit `DRIVE_ROOT` if you want a different folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/HPM'      # change if you like
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints/phaseA_celeba_contrastive'
SPLITS_DIR = f'{DRIVE_ROOT}/splits'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)
print('checkpoints →', CKPT_DIR)

## 3. Get the HPM code into Colab

The repo is not yet a public git repo, so the default path installs it from a
**zip you uploaded to Drive**. To use it: zip the repo locally (the folder that
contains `pyproject.toml`) and put the zip at `DRIVE_ROOT/hpm_repo.zip`.

If you instead push the repo to GitHub, set `INSTALL_FROM = 'github'` and fill in
`GITHUB_URL`.

In [ ]:
INSTALL_FROM = 'drive'   # 'drive' | 'github'
GITHUB_URL   = 'https://github.com/<you>/HPM.git'
REPO_DIR     = '/content/HPM'

import shutil, subprocess
if INSTALL_FROM == 'github':
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', GITHUB_URL, REPO_DIR], check=True)
else:
    zip_path = f'{DRIVE_ROOT}/hpm_repo.zip'
    assert os.path.exists(zip_path), f'upload the repo zip to {zip_path}'
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    shutil.unpack_archive(zip_path, '/content/_hpm_unzip')
    # find the folder containing pyproject.toml
    import glob
    root = next(p for p in glob.glob('/content/_hpm_unzip/**/pyproject.toml', recursive=True))
    shutil.move(os.path.dirname(root), REPO_DIR)
print('repo at', REPO_DIR)

## 4. Install dependencies

In [ ]:
!pip install -q -e "{REPO_DIR}[dev]"
!pip install -q kaggle

## 5. Download CelebA via the Kaggle API

Upload your `kaggle.json` (Kaggle → Account → Create New API Token) when prompted.
Downloads `jessicali9530/celeba-dataset` and unzips the aligned crops + identity file.

In [ ]:
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload kaggle.json:')
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'wb') as f:
        f.write(next(iter(up.values())))
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

DATA_ROOT = '/content/data/celeba'   # local SSD — fast; re-download on reconnect
os.makedirs(DATA_ROOT, exist_ok=True)
if not os.path.exists(f'{DATA_ROOT}/identity_CelebA.txt') and not os.path.exists(f'{DATA_ROOT}/list_identity_celeba.txt'):
    !kaggle datasets download -d jessicali9530/celeba-dataset -p {DATA_ROOT}
    !unzip -q {DATA_ROOT}/celeba-dataset.zip -d {DATA_ROOT}
!ls {DATA_ROOT}

In [ ]:
# The Kaggle distribution names the identity file `list_identity_celeba.txt` and
# prefixes a header/count. Normalise it to the `identity_CelebA.txt` the loader expects.
import os, glob
src = None
for name in ['identity_CelebA.txt', 'list_identity_celeba.txt']:
    hits = glob.glob(f'{DATA_ROOT}/**/{name}', recursive=True)
    if hits:
        src = hits[0]; break
assert src, 'identity file not found'
dst = f'{DATA_ROOT}/identity_CelebA.txt'
lines = open(src).read().splitlines()
# Drop a leading count line and/or a header line if present (keep only 'fname id').
rows = [ln for ln in lines if len(ln.split()) == 2 and ln.split()[1].isdigit()]
open(dst, 'w').write('\n'.join(rows) + '\n')
print('wrote', dst, '—', len(rows), 'images')

## 6. Compose the config

Loads the repo's Hydra configs and overrides the data root, Drive checkpoint/split
paths, and seed. Switches the experiment to the contrastive recipe.

In [ ]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf

SEED = 42
with initialize_config_dir(config_dir=f'{REPO_DIR}/configs', version_base=None):
    cfg = compose(
        config_name='config',
        overrides=[
            'data=celeba',
            'experiment=phaseA_celeba_contrastive',
            f'data.root={DATA_ROOT}',
            f'data.splits_dir={SPLITS_DIR}',
            f'train.checkpoint.dir={CKPT_DIR}',
            f'train.seed={SEED}',
        ],
    )
print(OmegaConf.to_yaml(cfg))

## 7. (Optional) Build + sanity-check identity splits

The trainer builds and persists splits automatically, but running this first lets
you confirm the train/val/test identity sets are **disjoint** (no identity leakage).

In [ ]:
from pathlib import Path
from hpm.data.splits import make_splits

tr, va, te = make_splits(Path(cfg.data.root), cfg, cfg.train.seed)
sa = {l for _, l in tr}; sb = {l for _, l in va}; sc = {l for _, l in te}
print(f'train: {len(tr)} imgs / {len(sa)} ids | val: {len(va)}/{len(sb)} | test: {len(te)}/{len(sc)}')
assert sa.isdisjoint(sb) and sa.isdisjoint(sc) and sb.isdisjoint(sc), 'IDENTITY LEAKAGE!'
print('splits are identity-disjoint ✓')

## 8. Train (auto-resume)

Writes `last.pt` every epoch and `best.pt` whenever verification ROC-AUC improves.
**If Colab disconnects, just rerun this cell** — `resume='auto'` continues from
`last.pt`.

> Smoke test (doc §7.1): set `cfg.train.max_steps = 300` before running to confirm
> the loss drops and ROC-AUC > 0.5 in a few minutes.

In [ ]:
from hpm.train_contrastive import run

# cfg.train.max_steps = 300   # uncomment for a quick smoke test
summary = run(cfg, resume='auto')
print(summary)

## 9. Evaluate `best.pt`

Reloads the best checkpoint and reports verification metrics on held-out identities.
(Hook the behavioral-signature harness here once Milestone 0 eval is wired in.)

In [ ]:
import torch
from torch.utils.data import DataLoader
from hpm.train import _build_model_cfg
from hpm.models.hpm_model import HPMModel
from hpm.data.dataset import FaceDataset
from hpm.eval.verification import evaluate_verification
from hpm.utils.checkpoint import load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_cfg = _build_model_cfg(cfg); OmegaConf.set_struct(model_cfg, False)
tr, va, te = make_splits(Path(cfg.data.root), cfg, cfg.train.seed)
model_cfg.data.num_identities = len({l for _, l in tr})

model = HPMModel(model_cfg).to(device)
load_checkpoint(f'{CKPT_DIR}/best.pt', model=model, cfg=model_cfg, map_location=device, restore_rng=False)

test_loader = DataLoader(FaceDataset(te, model_cfg), batch_size=cfg.train.batch_size, num_workers=2)
for stream in ['z_F', 'z_L', 'z_R']:
    m = evaluate_verification(model, test_loader, device, which=stream, seed=cfg.train.seed)
    print(f'{stream}:  roc_auc={m["roc_auc"]:.4f}  tar@far={m["tar_at_far"]:.4f}  acc={m["accuracy"]:.4f}')